# Airline Chat Bot Exercise

### Importing packages

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

### Defining model variables

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
MODEL = "openai/gpt-oss-120b:free"
openai = OpenAI(base_url=os.getenv("BASE_URL"), api_key=openai_api_key)

### System Prompt

In [ ]:
system_message = """
    You are a helpful assistant for an Airline called FlightAI.
    Give short, courteous answers, no more than 1 sentence.
    Always be accurate. If you don't know the answer, say so.
"""

### Creating the `prices` table

In [ ]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

### Defining tool functions

#### Function to get price of a city

In [ ]:
def get_ticket_price(city):
    print(f"Getting price for {city}.")
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [ ]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

#### Function to set the price of a city

In [ ]:
def set_ticket_price(city, price):
    print(f"Setting price for {city} to USD {price}.")
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [ ]:
set_price_function = {
    "name": "set_ticket_price",
    "description": "Set the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
            "price": {
                "type": "number",
                "description": "The return ticket price",
            }
        },
        "required": ["city", "price"],
        "additionalProperties": False
    }
}

#### Function to clear the `prices` table

In [ ]:
def empty_table():
    print(f"Deleting all the rows in table prices.")
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('DELETE FROM prices')
        conn.commit()

In [ ]:
empty_table_function = {
    "name": "empty_table",
    "description": "Delete all rows from the prices table in the SQLite database.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False
    }
}

#### Function to print the prices of all the cities

In [ ]:
def get_all_ticket_prices():
    print("Getting all ticket prices.")

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT city, price FROM prices")
        rows = cursor.fetchall()

        if not rows:
            return "No cities in our records."

        # Build Markdown table
        md = "| City | Price |\n"
        md += "|------|-------|\n"

        for city, price in rows:
            md += f"| {city.title()} | ${price} |\n"

        return md

In [ ]:
get_all_ticket_prices_function = {
    "name": "get_all_ticket_prices",
    "description": "Retrieve all ticket prices from the database and return them in a Markdown table format. If no records exist, return a message indicating no cities are in the records.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False
    }
}

#### Adding tools into the bucket of tools

In [ ]:
tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": set_price_function},
    {"type": "function", "function": empty_table_function},
    {"type": "function", "function": get_all_ticket_prices_function}
]

### Scalable functions to call any tool without `if-else` statements

In [ ]:
def call_tool(function_name, **kwargs):
    func = globals().get(function_name)
    if not callable(func):
        raise ValueError(f"Function '{function_name}' not found")
    return func(**kwargs)

def handle_tool_switches(tool_call, responses):
    function_name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)
    tool_response = call_tool(function_name, **arguments)
    if tool_response is not None:
        responses.append({
            "role": "tool",
            "content": tool_response,
            "tool_call_id": tool_call.id
        })

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        handle_tool_switches(tool_call, responses)
    return responses

### Chat function

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

### Gradio interface

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()